In [1]:
import random
import sys
from time import time
from collections import deque, namedtuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import csv
import matplotlib.pyplot as plt
import pickle
import cvxpy as cp
import math

# ── Import the environment from its own module ────────────────────────────
from solar_microgrid_env import SolarMicrogridEnv

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda:0


In [2]:
# Load and solar data
df = pd.read_csv('Jan_hourly_data_combined_all_modified.csv')

solar_data_raw = df['ghi'].values
load_data      = df['demand'].values * 0.5          # fraction of total demand (kW)
price_data_raw = df['price'].values

# Convert solar irradiance → solar power
Pmax_PV = 3000      # 3 MW / 3000 kW
Rc      = 120       # Irradiance point (W/m²)
Gstd    = 1000      # Standard irradiance (W/m²)

solar_power = np.where(
    (solar_data_raw > 0) & (solar_data_raw < Rc),
    Pmax_PV * (solar_data_raw ** 2 / (Gstd * Rc)),
    Pmax_PV * (solar_data_raw / Gstd),
)

# Scale to MW
solar_data = solar_power / 1000
load_data  = load_data  / 1000
price_data = price_data_raw

In [3]:
# Discretised action spaces
DG1_space        = [0, 0.5, 1, 1.5, 2]
DG2_space        = [0, 1, 2, 3]
DG3_space        = [0, 1, 2, 3, 4]
PGrid_space      = [0, 2, 4, 6, 8]
DG1_status       = [0, 1]
DG2_status       = [0, 1]
DG3_status       = [0, 1]
Batt_Power_space = [-0.5, -0.3, -0.1, 0, 0.1, 0.3, 0.5]

# Instantiate environment (imported from solar_microgrid_env.py)
env = SolarMicrogridEnv(
    load_data, solar_data, price_data,
    DG1_space, DG2_space, DG3_space, PGrid_space,
    DG1_status, DG2_status, DG3_status, Batt_Power_space,
)

state_size  = env.state_size
action_size = env.action_size
print(f"state size: {state_size}, action size: {action_size}")

state size: 4, action size: 28000


In [4]:
MIN_EXPLORATION_PROB = 1e-14
INITIAL_EXPLORATION_PROB = 1.0
EXPLORATION_DECAY = 0.995

TAU = 1e-3
LR = 1e-4
GAMMA = 0.8
UPDATE_EVERY = 4
BUFFER_SIZE = int(1e6)
BATCH_SIZE = 128
FC1_SIZE = 256
FC2_SIZE = 256
MAX_EPISODES = 5000
MAX_STEPS = len(solar_data)
ENV_SOLVED = 200
PRINT_EVERY = 100

In [5]:
class DuelingDQN(nn.Module):
  def __init__(self, state_size, fc1_size, fc2_size, action_size, seed):
    super(DuelingDQN, self).__init__()
    self.seed = torch.manual_seed(seed)
    self.layers = nn.Sequential(
        nn.Linear(state_size, fc1_size),
        nn.ReLU(),
        nn.Linear(fc1_size, fc2_size),
        nn.ReLU()
    )

    self.value = nn.Linear(fc2_size, 1)
    self.advantage = nn.Linear(fc2_size, action_size)

  def forward(self, x):
    x = self.layers(x)
    v = self.value(x)
    a = self.advantage(x)

    return v, a

In [6]:
class ReplayMemory():
  def __init__(self, buffer_size, batch_size, seed):
    self.batch_size = batch_size
    self.seed = random.seed(seed)
    self.memory = deque(maxlen = buffer_size)
    self.experience = namedtuple("Experience", field_names = ["state", "action", "reward", "next_state", "done"])

  def add_experience(self, state, action, reward, next_state, done):
    experience = self.experience(state, action, reward, next_state, done)
    self.memory.append(experience)

  def sample(self):
    experiences = random.sample(self.memory, k = self.batch_size)

    states = torch.from_numpy(np.vstack([experience.state for experience in experiences if experience is not None])).float().to(device)
    actions = torch.from_numpy(np.vstack([experience.action for experience in experiences if experience is not None])).long().to(device)
    rewards = torch.from_numpy(np.vstack([experience.reward for experience in experiences if experience is not None])).float().to(device)
    next_states = torch.from_numpy(np.vstack([experience.next_state for experience in experiences if experience is not None])).float().to(device)
    dones = torch.from_numpy(np.vstack([experience.done for experience in experiences if experience is not None])).long().to(device)

    return (states, actions, rewards, next_states, dones)

  def __len__(self):
    return len(self.memory)

In [7]:
class DuelingDoubleDeepQAgent_Fixed():
  def __init__(self, state_size, fc1_size, fc2_size, action_size, seed = 0):
    self.state_size = state_size
    self.action_size = action_size
    self.seed = random.seed(seed)

    self.q_network = DuelingDQN(self.state_size, fc1_size, fc2_size, self.action_size, seed).to(device)
    self.target_network = DuelingDQN(self.state_size, fc1_size, fc2_size, self.action_size, seed).to(device)
    self.optimizer = optim.Adam(self.q_network.parameters())

    self.memory = ReplayMemory(BUFFER_SIZE, BATCH_SIZE, seed)
    self.timestep = 0

  def step(self, state, action, reward, next_state, done):
    self.memory.add_experience(state, action, reward, next_state, done)
    self.timestep += 1
    if self.timestep % UPDATE_EVERY == 0:
        if len(self.memory) > (BATCH_SIZE):
            sampled_experience = self.memory.sample()
            self.learn(sampled_experience)

  def learn(self, experiences):
    states, actions, rewards, next_states, dones = experiences

    v_output, a_output = self.q_network(states)
    v_output_next, a_output_next = self.q_network(next_states)
    v_target, a_target = self.target_network(next_states)

    Q_output = torch.add(v_output, (a_output - a_output.mean(dim = 1, keepdim = True))).gather(1, actions)
    action_values = torch.add(v_output_next, (a_output_next - a_output_next.mean(dim = 1, keepdim = True)))
    target_action_values = torch.add(v_target, (a_target - a_output.mean(dim = 1, keepdim = True)))

    max_action_indices = target_action_values.max(1)[1].unsqueeze(1).detach()
    max_action_values = action_values.gather(1, max_action_indices)
    Q_target = rewards + GAMMA * max_action_values * (1 - dones)

    loss = F.mse_loss(Q_output, Q_target)

    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()

    self.update_target_network(self.q_network, self.target_network)

  def update_target_network(self, source_network, target_network): # Soft-update for the target network
    for source_parameters, target_parameters in zip(source_network.parameters(), target_network.parameters()):
        target_parameters.data.copy_(TAU * source_parameters.data + (1 - TAU) * target_parameters.data)

  def act(self, state, eps = 0.0):
    rnd = random.random()
    if rnd < eps:
        return np.random.randint(self.action_size)
    else:
        state = np.array(state)
        state = torch.from_numpy(state).float().unsqueeze(0).to(device)

        self.q_network.eval() # Puts the network in evaluation mode. Disables gradient tracking
        with torch.no_grad():
            _, action_values = self.q_network(state)

        self.q_network.train() # Returns it back to the training mode
        action = np.argmax(action_values.cpu().data.numpy())
        return action

  def checkpoint(self, filename):
    torch.save(self.q_network.state_dict(), filename)

In [8]:
def calculate_bounds_gen_1(action):
    mapping = {0: (0,0), 0.5:(0,500), 1:(500,1000), 1.5:(1000,1500), 2:(1500,2000)}
    return mapping.get(action, ('NaN','NaN'))

def calculate_bounds_gen_2(action):
    mapping = {0:(0,0), 1:(0,1000), 2:(1000,2000), 3:(2000,3000)}
    return mapping.get(action, ('NaN','NaN'))

def calculate_bounds_gen_3(action):
    mapping = {0:(0,0), 1:(0,1000), 2:(1000,2000), 3:(2000,3000), 4:(3000,4000)}
    return mapping.get(action, ('NaN','NaN'))

def calculate_bounds_grid(action):
    mapping = {0:(0,0), 2:(0,2000), 4:(2000,4000), 6:(4000,6000), 8:(6000,8000)}
    return mapping.get(action, ('NaN','NaN'))

def calculate_bounds_batt(action):
    mapping = {-0.5:(-500,-300), -0.3:(-300,-100), -0.1:(-100,0),
                0:(0,0), 0.1:(0,100), 0.3:(100,300), 0.5:(300,500)}
    return mapping.get(action, ('NaN','NaN'))


def _solve_qp(lb_pgs, ub_pgs, lb_pgrid, ub_pgrid, lb_pbatt, ub_pbatt,
              a_coefs, b_coefs, c_coefs, pg_status, solar_power,
              load, price, batt_energy_set, bess_coef, pv_omcosts):
    """Shared QP core used by both train and test refinement."""
    pg1, pg2, pg3, pgrid, pbatt = cp.Variable(), cp.Variable(), cp.Variable(), cp.Variable(), cp.Variable()
    a1, a2, a3 = a_coefs
    xg1, xg2, xg3 = pg_status
    b1, b2, b3 = b_coefs
    c1, c2, c3 = c_coefs
    uchg = 1 if lb_pbatt <= 0 and ub_pbatt <= 0 else 0

    gen_costs   = (xg1*((a1*pg1**2)+(b1*pg1)+c1)
                 + xg2*((a2*pg2**2)+(b2*pg2)+c2)
                 + xg3*((a3*pg3**2)+(b3*pg3)+c3))
    grid_cost   = price * pgrid
    batt_cost   = (uchg*(-bess_coef)*pbatt) + ((1-uchg)*bess_coef*pbatt) + (bess_coef*batt_energy_set)
    pv_cost     = pv_omcosts * solar_power

    objective   = cp.Minimize(gen_costs + grid_cost + batt_cost + pv_cost)
    constraints = (
        [pg1+pg2+pg3+pgrid+solar_power-pbatt >= load]
        + [pg1 >= lb_pgs[0]/1000, pg2 >= lb_pgs[1]/1000, pg3 >= lb_pgs[2]/1000,
           pgrid >= lb_pgrid/1000, pbatt >= lb_pbatt/1000]
        + [pg1 <= ub_pgs[0]/1000, pg2 <= ub_pgs[1]/1000, pg3 <= ub_pgs[2]/1000,
           pgrid <= ub_pgrid/1000, pbatt <= ub_pbatt/1000]
    )
    prob = cp.Problem(objective, constraints)
    prob.solve()

    if prob.status == 'optimal':
        return (round(float(pg1.value),1), round(float(pg2.value),1),
                round(float(pg3.value),1), round(float(pgrid.value),1),
                round(float(pbatt.value),1), prob.value)
    else:
        print("No optimal solution found.")
        return (ub_pgs[0]/1000, ub_pgs[1]/1000, ub_pgs[2]/1000,
                ub_pgrid/1000, ub_pbatt/1000, 1_000_000)

# Convenience aliases (kept for backward compatibility)
def refine_variables(*args, **kwargs):
    return _solve_qp(*args, **kwargs)

def refine_variables_test(*args, **kwargs):
    return _solve_qp(*args, **kwargs)

In [9]:
def print_and_save_to_csv(output, filename):
    print(output)
    with open(filename, 'a', newline='') as f:
        csv.writer(f).writerow([output])

In [10]:
def train(dqn_agent, action_space):
  start = time()
  scores = []
  uc_gen1, uc_gen2, uc_gen3, output_dg1, output_dg2, output_dg3, output_grid, cost_total, \
            batt_wh, batt_pwr, s_state, l_state, p_state, nbatt_state, diff_costs = [], [], [], [], [], [], [], [], [], [], [], [], [], [], []
  scores_window = deque(maxlen = 100)

  penalty_dict = {0: "Gen 1 status/output mismatch", 1: "Gen 2 status/output mismatch", 2: "Gen 3 status/output mismatch", 3: "Gen 1 Ramp limit", 4: "Gen 2 Ramp limit", \
                  5: "Gen 3 Ramp limit", 6: "Battery SoC limit", 7: "Power balance limit", 8: "Supply < Demand & Battery Power is zero",\
                  9: "Ramp limit G1 @ start", 10: "Ramp limit G2 @ start", 11: "Ramp limit G3 @ start", 12: "Min downtime of G1", 13: "Min downtime of G2", 14: "Min downtime of G3", \
                  15: "Min uptime of G1", 16: "Min uptime of G2", 17: "Min uptime of G3"}

  # Variables to store more plots results (costs, penalties violated)
  all_costs = []; tot_violated_penalties = []

  for episode in range(MAX_EPISODES):
    init_prod_dg1 = 1; init_prod_dg2 = 1; init_prod_dg3 = 1           # Initial production from generators
    state = env.reset()
    state = np.array(state)
    score = 0
    score_cost = 0
    score_penalties = 0

    for t in range(MAX_STEPS):
      eps = max(MIN_EXPLORATION_PROB, INITIAL_EXPLORATION_PROB * (EXPLORATION_DECAY**episode))
      soc_prev = state[3]
      action = dqn_agent.act(state, eps)
      action_elements = action_space[action]

      # Extract corresponding DG output and power traded with main grid.
      dg1_action = action_elements[0]; dg1_status_action = action_elements[1]; dg2_action = action_elements[2]; dg2_status_action = action_elements[3]; \
      dg3_action = action_elements[4]; dg3_status_action = action_elements[5]; pgrid_action = action_elements[6]; decision_pbatt = action_elements[7]

      # Extract elements
      next_state, reward, obj, all_diff_costs, penalty_total, done = env.step(t, init_prod_dg1, init_prod_dg2, init_prod_dg3, dg1_action, dg2_action, dg3_action, \
                                                    pgrid_action, dg1_status_action, dg2_status_action, dg3_status_action, decision_pbatt, soc_prev)

      # Battery energy to use for variable refinement
      batt_energy_set = next_state[3]

      if episode == (MAX_EPISODES - 1):
            # Count the number of constraints violated
            violated_constraints = sum(1 for value in penalty_total if value > 0)

            print("\nRaw outputs:\n")
            output_raw = f"Battery Energy: {batt_energy_set}, DGs output/status: {(dg1_action, dg1_status_action), (dg2_action, dg2_status_action), (dg3_action, dg3_status_action)}, Grid: {pgrid_action}, Battery Power: {decision_pbatt}, Solar: {state[0]}, Load: {state[1]}, \
                Obj: {obj}, Violated Constraints: {violated_constraints}\n"

            # Print output and save same to a CSV file
            print_and_save_to_csv(output_raw, "raw_outputs_A_duelingddqn.csv")

            # PRINT VIOLATED CONSTRAINTS!
            output_string = ""
            with open("raw_penalty_outputs_A_duelingddqn.csv", 'a', newline='') as csvfile:
              writer = csv.writer(csvfile)
              if violated_constraints == 0:
                output_string += "No Violation, "
              # Iterate through the penalty list
              for idx, value in enumerate(penalty_total):
                if value > 0:
                  output_penalty = f"{penalty_dict[idx]}"
                  output_string += output_penalty + ", "
              # Write the output string to the CSV file
              writer.writerow([output_string.rstrip(", ")])
              print(f'Violated constraints: {output_string.rstrip(", ")}')

            # Store data for plotting
            uc_gen1.append(dg1_status_action)
            uc_gen2.append(dg2_status_action)
            uc_gen3.append(dg3_status_action)

            output_dg1.append(dg1_action)
            output_dg2.append(dg2_action)
            output_dg3.append(dg3_action)
            output_grid.append(pgrid_action)

            s_state.append(state[0])  # appending solar power
            l_state.append(state[1])  # appending load
            p_state.append(state[2])  # appending price
            nbatt_state.append(batt_energy_set)  # appending battery energy of next state

            cost_total.append(obj)
            diff_costs.append(all_diff_costs)  # DG costs, Grid costs, Battery degradation costs, PV O&M costs

            batt_wh.append(soc_prev)
            batt_pwr.append(decision_pbatt)

      dqn_agent.step(state, action, reward, next_state, done)
      state = next_state
      init_prod_dg1 = dg1_action; init_prod_dg2 = dg2_action; init_prod_dg3 = dg3_action
      score += reward
      score_cost += obj
      count_violated_constraints = sum(1 for value in penalty_total if value > 0)
      score_penalties += count_violated_constraints

      if done:
        break

    scores_window.append(score)
    scores.append(score)
    all_costs.append(score_cost)
    tot_violated_penalties.append(score_penalties)

    if episode % PRINT_EVERY == 0:
      mean_score = np.mean(scores_window)
      print('\r Progress {}/{}, average score: {:.2f}, epsilon: {}'.format(episode, MAX_EPISODES, mean_score, eps), end = "")

    if episode == (MAX_EPISODES - 1):
      dqn_agent.checkpoint('solved_200_T3_duelingddqn.pth')
      print(f"\n Episode completed...now proceeding to save DRL model")
      break

  end = time()
  print('\n Took {} seconds'.format(end - start))

  return scores, all_costs, tot_violated_penalties, uc_gen1, uc_gen2, uc_gen3, output_dg1, output_dg2, output_dg3, \
          output_grid, cost_total, batt_wh, batt_pwr, s_state, l_state, p_state, nbatt_state, diff_costs

In [ ]:
duelingddqn_agent = DuelingDoubleDeepQAgent_Fixed(state_size, FC1_SIZE, FC2_SIZE, action_size, seed = 0)

duelingddqn_scores, all_costs, tot_violated_penalties, uc_gen1, uc_gen2, uc_gen3, output_dg1, output_dg2, output_dg3, \
              output_grid, cost_total, batt_wh, batt_pwr, s_state, l_state, p_state, nbatt_state, diff_costs = train(duelingddqn_agent, env.action_space)

In [ ]:
# Raw cost function (Train set)
train_obj_raw = sum(cost_total)
print(f"Total cost function (Raw Train Set): {train_obj_raw}" )

### Augmented DQN — Quadratic Programming Refinement (Train)

In [ ]:
a_coefs  = [0.01, 0.02, 0.03]
b_coefs  = [0.55, 0.75, 0.85]
c_coefs  = [1,    2.5,  3.3 ]

bess_rated = 1.5
bess_coef  = (80 * bess_rated) / (bess_rated * 3000)
pv_omcosts = 0.7

new_dg1_action, new_dg2_action, new_dg3_action = [], [], []
new_pgrid_action, new_pbatt_action, new_obj, new_diff_costs = [], [], [], []

for t in range(MAX_STEPS):
    sp = s_state[t]; ld = l_state[t]; pr = p_state[t]; be = nbatt_state[t]
    pg_status = [uc_gen1[t], uc_gen2[t], uc_gen3[t]]

    lb_pgs  = [calculate_bounds_gen_1(output_dg1[t])[0], calculate_bounds_gen_2(output_dg2[t])[0], calculate_bounds_gen_3(output_dg3[t])[0]]
    ub_pgs  = [calculate_bounds_gen_1(output_dg1[t])[1], calculate_bounds_gen_2(output_dg2[t])[1], calculate_bounds_gen_3(output_dg3[t])[1]]
    lb_pgrid, ub_pgrid = calculate_bounds_grid(output_grid[t])
    lb_pbatt, ub_pbatt = calculate_bounds_batt(batt_pwr[t])

    d1, d2, d3, dg, db, mo = refine_variables(
        lb_pgs, ub_pgs, lb_pgrid, ub_pgrid, lb_pbatt, ub_pbatt,
        a_coefs, b_coefs, c_coefs, pg_status, sp, ld, pr, be, bess_coef, pv_omcosts,
    )

    rc_dg   = (pg_status[0]*((a_coefs[0]*d1**2)+(b_coefs[0]*d1)+c_coefs[0])
             + pg_status[1]*((a_coefs[1]*d2**2)+(b_coefs[1]*d2)+c_coefs[1])
             + pg_status[2]*((a_coefs[2]*d3**2)+(b_coefs[2]*d3)+c_coefs[2]))
    rc_grid = pr * dg
    rc_batt = (bess_coef * abs(db)) + (bess_coef * be)
    rc_pv   = pv_omcosts * sp

    new_dg1_action.append(d1); new_dg2_action.append(d2); new_dg3_action.append(d3)
    new_pgrid_action.append(dg); new_pbatt_action.append(db); new_obj.append(mo)
    new_diff_costs.append((rc_dg, rc_grid, rc_batt, rc_pv))

with open('refined_complete_result_augment_T3_dqn.pickle', 'wb') as f:
    pickle.dump([new_dg1_action, new_dg2_action, new_dg3_action,
                 new_pgrid_action, new_pbatt_action, new_obj, new_diff_costs], f)

print(f"Total cost function (Refined Train Set): {sum(new_obj)}")

### Saving and reloading all results to a pickle file

In [ ]:
# Putting all results in a list
duelingddqn_all_results = [duelingddqn_scores, all_costs, tot_violated_penalties, uc_gen1, uc_gen2, uc_gen3, output_dg1, output_dg2, output_dg3, \
                          output_grid, cost_total, batt_wh, batt_pwr, s_state, l_state, p_state, nbatt_state, diff_costs]

with open('complete_result_augment_T3_duelingddqn.pickle', 'wb') as file:
  pickle.dump(duelingddqn_all_results, file)
  print('All results file saved successfully!')

In [ ]:
# Load the saved pickle file
with open('complete_result_augment_T3_duelingddqn.pickle', 'rb') as file:
  duelingddqn_all_results = pickle.load(file)
  print('All results file loaded successfully!')

### Plotting Results

In [ ]:
def plot(score, title, ylabel):
    fig, ax = plt.subplots(figsize=(15, 10), dpi=300)
    ax.plot(score)
    ax.plot(pd.Series(score).rolling(100).mean(), linewidth=4)
    ax.set_title(title, fontsize=20); ax.set_xlabel('Episodes', fontsize=20)
    ax.set_ylabel(ylabel, fontsize=20); ax.grid(True)
    plt.savefig(f'{ylabel}.png', dpi=300, bbox_inches='tight')
    plt.show()

plot(duelingddqn_all_results[0][:-1], 'Augmented DQN Algorithm', 'Reward')
plot(duelingddqn_all_results[1][:-1], 'Aggregated Costs (DQN)',   'AUD ($) per episode')
plot(duelingddqn_all_results[2][:-1], 'Total Violated Constraints (TVCs)', 'TVCs per episode')

### Test the Model

In [12]:
import pickle, pandas as pd

with open('scenario_demand_data_2020.pickle', 'rb') as f:
    load_demand = pickle.load(f)
with open('scenario_price_data_2020.pickle', 'rb') as f:
    price_test = pickle.load(f)
with open('scenario_solar_data_2020.pickle', 'rb') as f:
    solar_irradiance = pickle.load(f)

# Fill solar to a full 24-h index
idx  = pd.date_range('2020-01-15 06:00', '2020-01-15 19:00', freq='h')
df_s = pd.DataFrame(solar_irradiance, index=idx)
full_idx = pd.date_range('2020-01-15 00:00', '2020-01-15 23:00', freq='h')
solar_irradiance = df_s.reindex(full_idx, fill_value=0)

load_demand      = load_demand['prediction'].values * 0.5
price_test       = price_test['prediction'].values
solar_irradiance = solar_irradiance['prediction'].values

# Irradiance → power
solar_power_test = np.where(
    (solar_irradiance > 0) & (solar_irradiance < Rc),
    Pmax_PV * (solar_irradiance ** 2 / (Gstd * Rc)),
    Pmax_PV * (solar_irradiance / Gstd),
)

MAX_STEPS_TEST = 24
solar_power_test = solar_power_test / 1000
load_demand      = load_demand      / 1000

env_test = SolarMicrogridEnv(
    load_demand, solar_power_test, price_test,
    DG1_space, DG2_space, DG3_space, PGrid_space,
    DG1_status, DG2_status, DG3_status, Batt_Power_space,
)

In [13]:
model_path = "solved_200_T3_duelingddqn.pth"
model      = DuelingDQN(state_size, FC1_SIZE, FC2_SIZE, action_size, seed=0)
model.load_state_dict(torch.load(model_path, weights_only=True))
model.eval().to(device)

test_dqn_agent              = DuelingDoubleDeepQAgent_Fixed(state_size, FC1_SIZE, FC2_SIZE, action_size, seed=0)
test_dqn_agent.q_network    = model

In [14]:
def test(test_agent, action_space, load_demand, solar_power):
    MAX_STEPS_T = len(load_demand)
    start = time()
    scores  = []
    tuc_gen1, tuc_gen2, tuc_gen3 = [], [], []
    toutput_dg1, toutput_dg2, toutput_dg3, toutput_grid = [], [], [], []
    tcost_total, tbatt_wh, tbatt_pwr = [], [], []
    ts_state, tl_state, tp_state, tnbatt_state, tdiff_costs = [], [], [], [], []

    penalty_dict = {
        0: "Gen 1 status/output mismatch", 1: "Gen 2 status/output mismatch",
        2: "Gen 3 status/output mismatch", 3: "Gen 1 Ramp limit",
        4: "Gen 2 Ramp limit",             5: "Gen 3 Ramp limit",
        6: "Battery SoC limit",            7: "Power balance limit",
        8: "Supply < Demand & Battery Power is zero",
        9: "Ramp limit G1 @ start",       10: "Ramp limit G2 @ start",
       11: "Ramp limit G3 @ start",       12: "Min downtime of G1",
       13: "Min downtime of G2",          14: "Min downtime of G3",
       15: "Min uptime of G1",            16: "Min uptime of G2",
       17: "Min uptime of G3",
    }

    bess_rated = 1.5
    bess_coef  = (80 * bess_rated) / (bess_rated * 3000)
    pv_omcosts = 0.7

    init_prod_dg1 = init_prod_dg2 = init_prod_dg3 = 1
    state = np.array(env_test.reset())
    score = 0

    for t in range(MAX_STEPS_T):
        soc_prev = state[3]
        action   = test_agent.act(state, eps=0)
        a        = action_space[action]
        dg1_action, dg1_status_action = a[0], a[1]
        dg2_action, dg2_status_action = a[2], a[3]
        dg3_action, dg3_status_action = a[4], a[5]
        pgrid_action, decision_pbatt  = a[6], a[7]

        next_state, reward, obj, all_diff_costs, penalty_total, done = env_test.step(
            t, init_prod_dg1, init_prod_dg2, init_prod_dg3,
            dg1_action, dg2_action, dg3_action, pgrid_action,
            dg1_status_action, dg2_status_action, dg3_status_action,
            decision_pbatt, soc_prev,
        )
        batt_energy_set = next_state[3]
        violated        = sum(1 for v in penalty_total if v > 0)

        raw_out = (f"Solar: {solar_power[t]}, Load: {load_demand[t]}, Price: {price_test[t]}, "
                   f"DGs: {(dg1_action,dg2_action,dg3_action)}, Grid: {pgrid_action}, "
                   f"BESS: {decision_pbatt}, BattE: {batt_energy_set}, Obj: {obj}")
        print_and_save_to_csv(raw_out, "test_raw_outputs_A_dqn.csv")

        out_str = ""
        with open("test_penalty_outputs_A_dqn.csv", 'a', newline='') as f:
            w = csv.writer(f)
            if violated == 0:
                out_str += "No Violation, "
            for idx, val in enumerate(penalty_total):
                if val > 0:
                    out_str += penalty_dict[idx] + ", "
            w.writerow([out_str.rstrip(", ")])
        print(f'Violated: {out_str.rstrip(", ")}')

        tuc_gen1.append(dg1_status_action); tuc_gen2.append(dg2_status_action); tuc_gen3.append(dg3_status_action)
        toutput_dg1.append(dg1_action); toutput_dg2.append(dg2_action); toutput_dg3.append(dg3_action)
        toutput_grid.append(pgrid_action)
        ts_state.append(solar_power[t]); tl_state.append(load_demand[t]); tp_state.append(price_test[t])
        tnbatt_state.append(batt_energy_set); tcost_total.append(obj)
        tbatt_wh.append(soc_prev); tbatt_pwr.append(decision_pbatt); tdiff_costs.append(all_diff_costs)

        state = next_state
        init_prod_dg1 = dg1_action; init_prod_dg2 = dg2_action; init_prod_dg3 = dg3_action
        score += reward

    print(f"Total test reward: {score}")
    return (score, tuc_gen1, tuc_gen2, tuc_gen3,
            toutput_dg1, toutput_dg2, toutput_dg3, toutput_grid,
            tcost_total, tbatt_wh, tbatt_pwr,
            ts_state, tl_state, tp_state, tnbatt_state, tdiff_costs)

In [ ]:
action_space_test = env_test.action_space
t0 = time()
(test_dqn_scores, tuc_gen1, tuc_gen2, tuc_gen3,
 toutput_dg1, toutput_dg2, toutput_dg3, toutput_grid,
 tcost_total, tbatt_wh, tbatt_pwr,
 ts_state, tl_state, tp_state, tnbatt_state, tdiff_costs) = test(
     test_dqn_agent, action_space_test, load_demand, solar_power_test
 )
print(f"Test elapsed: {time()-t0:.1f}s")
print(f"Total cost (Raw Test): {sum(tcost_total)}")

### Augmented Test Output Refinement (Quadratic Programming)

In [ ]:
a_coefs = [0.01, 0.02, 0.03]; b_coefs = [0.55, 0.75, 0.85]; c_coefs = [1, 2.5, 3.3]
bess_coef = (80 * 1.5) / (1.5 * 3000); pv_omcosts = 0.7

tnew_dg1, tnew_dg2, tnew_dg3, tnew_pgrid, tnew_pbatt, tnew_obj, tnew_diff = [], [], [], [], [], [], []

for t in range(MAX_STEPS_TEST):
    sp = ts_state[t]; ld = tl_state[t]; pr = tp_state[t]; be = tnbatt_state[t]
    pg_status = [tuc_gen1[t], tuc_gen2[t], tuc_gen3[t]]

    lb_pgs  = [calculate_bounds_gen_1(toutput_dg1[t])[0], calculate_bounds_gen_2(toutput_dg2[t])[0], calculate_bounds_gen_3(toutput_dg3[t])[0]]
    ub_pgs  = [calculate_bounds_gen_1(toutput_dg1[t])[1], calculate_bounds_gen_2(toutput_dg2[t])[1], calculate_bounds_gen_3(toutput_dg3[t])[1]]
    lb_pgrid, ub_pgrid = calculate_bounds_grid(toutput_grid[t])
    lb_pbatt, ub_pbatt = calculate_bounds_batt(tbatt_pwr[t])

    d1, d2, d3, dg, db, mo = refine_variables_test(
        lb_pgs, ub_pgs, lb_pgrid, ub_pgrid, lb_pbatt, ub_pbatt,
        a_coefs, b_coefs, c_coefs, pg_status, sp, ld, pr, be, bess_coef, pv_omcosts,
    )
    rc = (pg_status[0]*((a_coefs[0]*d1**2)+(b_coefs[0]*d1)+c_coefs[0])
        + pg_status[1]*((a_coefs[1]*d2**2)+(b_coefs[1]*d2)+c_coefs[1])
        + pg_status[2]*((a_coefs[2]*d3**2)+(b_coefs[2]*d3)+c_coefs[2]))

    tnew_dg1.append(d1); tnew_dg2.append(d2); tnew_dg3.append(d3)
    tnew_pgrid.append(dg); tnew_pbatt.append(db); tnew_obj.append(mo)
    tnew_diff.append((rc, pr*dg, bess_coef*abs(db)+bess_coef*be, pv_omcosts*sp))

with open('test_refined_complete_result_augment_T3_dqn.pickle', 'wb') as f:
    pickle.dump([tnew_dg1, tnew_dg2, tnew_dg3, tnew_pgrid, tnew_pbatt, tnew_obj, tnew_diff], f)

print(f"Total cost (Refined Test): {sum(tnew_obj)}")

In [ ]:
all_test_results = [test_dqn_scores, tuc_gen1, tuc_gen2, tuc_gen3,
                    toutput_dg1, toutput_dg2, toutput_dg3, toutput_grid,
                    tcost_total, tbatt_wh, tbatt_pwr,
                    ts_state, tl_state, tp_state, tnbatt_state, tdiff_costs]

with open('complete_test_result_augment_T3_dqn.pickle', 'wb') as f:
    pickle.dump(all_test_results, f)
    print('Test results saved!')